# Code pour l'intégration des données complétude MAPEPI

**Auteur** : Marie-Julie Lambert & Alex Kaldjian 

**Date** : Janvier 2022

**Projet** : Macepa (Gates)

**Pays** : RDC

  
**Mises à jour du processus de chargement :** Esteban Montandon (février 2026)

## Chargement des librairies et connexion à la base de données PostGres

In [ ]:
install.packages('stringdist', quiet = TRUE)
library(stringdist)

library(stringr)
library(readxl)
library(data.table)
library(sf)
library(DBI)

In [ ]:
con <- dbConnect(
    RPostgres::Postgres(),
    dbname = Sys.getenv("WORKSPACE_DATABASE_DB_NAME"),
    host = Sys.getenv("WORKSPACE_DATABASE_HOST"),
    port = Sys.getenv("WORKSPACE_DATABASE_PORT"),
    user = Sys.getenv("WORKSPACE_DATABASE_USERNAME"),
    password = Sys.getenv("WORKSPACE_DATABASE_PASSWORD")
)

### Chargement des données depuis le nouveau fichier

In [ ]:
# nouvelle_completude_path <- "/home/hexa/workspace/tdb-suivi-epidemio/data/raw/Completude/2024/COMPLETUDE PROMPTITUDE SURVEILLANCE_Week11.xlsx"
# nouvelle_completude_path <- "/home/hexa/workspace/tdb-suivi-epidemio/data/raw/Completude/2025/COMPLETUDE PROMPTITUDE SURVEILLANCE_Week01.xlsx"
# nouvelle_completude_path <- "/home/hexa/workspace/tdb-suivi-epidemio/data/raw/Completude/2026/COMPLETUDE PROMPTITUDE SURVEILLANCE_Week05.xlsx"

In [ ]:
# Completude data path
raw_data_path <- "/home/hexa/workspace/tdb-suivi-epidemio/data/raw/Completude/"

In [ ]:
read_helper <- function(sheet, file) {
    data <- suppressMessages(
        setDT(
            read_excel(
                file,
                sheet
            )
        )
    )
    
    title <- colnames(data)[1]
    
    names(data) <- as.character(data[1,])
    data <- data[-1, ]

    # exception handling for mislabeled provinces
    data <- tryCatch( {
        data <- data[`PROVINCE/DPS` != "TOTAL"]  # data[data[,1] != "TOTAL"] 
        },
    
    error = function(cond) {
        print(paste('Nom de la colonne Province mal formaté pour la semaine',
                    sheet))
        
        names(data)[1] <- "PROVINCE/DPS"
        data <- data[`PROVINCE/DPS` != "TOTAL"]
        }
    )
    
    # data[, sheet_name := sheet]
    data[, sheet_title := title]

    return(data)
}

extract_year_from_path <- function(path) {
  file_name <- basename(path)
  # Check if filename starts with 4 digits
  if (!grepl("^[0-9]{4}", file_name)) {
    return(NULL)
  }
  
  year <- as.numeric(sub("^([0-9]{4}).*", "\\1", file_name))
  
  # Extra safety check (in case conversion fails)
  if (is.na(year)) {
    return(NULL)
  }  
  return(year)
}

# Function to get the latest week file from a folder
get_latest_week_file <- function(folder_path, pattern = "_Week([0-9]+)\\.csv$") {
  all_files <- list.files(folder_path, full.names = TRUE)
  
  # Filter files matching the week pattern
  week_files <- all_files[str_detect(all_files, pattern)]
  
  # If no files found, return NULL
  if(length(week_files) == 0){
    warning("No files matching the pattern found in folder: ", folder_path)
    return(NULL)
  }
  
  # Extract week numbers
  week_numbers <- str_extract(week_files, pattern) %>% 
    str_extract("[0-9]+") %>% 
    as.integer()
  
  # Return the file with the max week number
  week_files[which.max(week_numbers)]
}

### 1.1 Load current year data

In [ ]:
## NEW CODE
nouvelle_completude_path

In [ ]:
## Read current year data
current_year <- as.numeric(basename(dirname(nouvelle_completude_path)))
if (is.null(current_year) | is.na(current_year)) stop(glue::glue("ERROR: Year can not be identified in path: {nouvelle_completude_path}"))

sheet.names <- as.list(excel_sheets(nouvelle_completude_path))

# remove sheets that are improperly named
sheet.names <- sheet.names[!(sheet.names %like% "Feuil")]

# Update variable Name !
dse_current <- rbindlist(lapply(sheet.names, read_helper, file = nouvelle_completude_path), use.names = FALSE)

# regex robust to both S10 and SE10 b/c they can't keep a consistent format
dse_current[, NUMSEM := as.integer(gsub("\\D", "", sheet_title))]
colnames(dse_current)[1] <- c("Province")
dse_current[, year := current_year] 

### 1.2 Load previous years data

In [ ]:
# Get the list of completude folders per year
all_dirs <- list.dirs(raw_data_path, full.names = TRUE)
year_dirs <- all_dirs[grepl("/[0-9]{4}$", all_dirs)]
year_dirs <- year_dirs[!grepl(current_year, year_dirs)]  # Exclude current year!
year_dirs

In [ ]:
comp_prev_years <- data.table()
for (year_dir in year_dirs) {
    
    year_folder <- as.numeric(basename(year_dir))
    if (is.null(year_folder) | is.na(year_folder)) stop(glue::glue("ERROR: Year can not be identified in path: {year_dir}"))
    
    completude_path <- get_latest_week_file(year_dir, pattern = "_Week([0-9]+)\\.xlsx$")
    if (is.null(completude_path)) stop(glue::glue("ERROR: Unable to find the latest file from : {year_dir}"))
    print(glue::glue("Chargement des données pour l’année {year_folder}: {completude_path} "))

    sheet.names <- as.list(excel_sheets(completude_path)) 
    sheet.names <- sheet.names[!(sheet.names %like% "Feuil")] # remove sheets that are improperly named
    completude_df <- rbindlist(lapply(sheet.names, read_helper, file = completude_path), use.names = FALSE)
    
    # regex robust to both S10 and SE10 b/c they can't keep a consistent format
    completude_df[, NUMSEM := as.integer(gsub("\\D", "", sheet_title))]
    colnames(completude_df)[1] <- c("Province")    
    completude_df[, year := year_folder] 

    # print(colnames(completude_df))

    comp_prev_years <- rbindlist(list(comp_prev_years, completude_df), use.names = FALSE)
    # comp_prev_years <- rbind(comp_prev_years, completude_df, fill = TRUE)
}

In [ ]:
df <- rbindlist(list(comp_prev_years, dse_current), use.names = FALSE)
unique(df$year)

### Alignement des noms des provinces

In [ ]:
ref <- st_read(
  paste0(
    "/home/hexa/workspace/tdb-suivi-epidemio/data/geodata/",
    "org_units-2021-10-21-14-55_level_province.gpkg"
  )
)

setDT(ref)

In [ ]:
pick_closest_name <- function(x, ref) {
    
    l <- stringdist(x, ref$name, method = 'jw')
    match <- ref$name[which(l == min(l))][[1]]
    
    return(match)
}

In [ ]:
df[, Province := gsub("ï", "i", Province)]
df[, Province := gsub("-", " ", Province)]


df[, province_ref := sapply(Province, pick_closest_name, ref = ref)]

In [ ]:
# still gotta fix a few
df[Province == 'Tshopo',  province_ref := "tp Tshopo Province"]
df[Province == 'Kasai', province_ref := "ks Kasai Province"]
df[Province == "Lomami", province_ref := "lm Lomami Province"]
df[Province == "KINSHASA", province_ref := "kn Kinshasa Province"]

# df[, sheet_name := NULL]
df[, sheet_title := NULL]

#df <- df[, .SD, .SDcols = names(df)[!(names(df) %like% 'AS')]]

# fix NAs
numeric.cols <- names(df)[!(names(df) %like% '.rovince')]
df[, (numeric.cols) := lapply(.SD, as.numeric), .SDcols = numeric.cols]

df[is.na(df)] <- 0

### Calcul de complétude des provinces

In [ ]:
df[,`PROMP/COMPL DPS` := `Rapport recu de la DPS` / `Rapport attendu de la DPS`]

In [ ]:
df[, NUMSEM := as.integer(NUMSEM)]
df[, year := as.integer(year)]

In [ ]:
tail(df)

### Chargement des données sur la DB

In [ ]:
dbWriteTable(con, "DSE_Completude", df, overwrite=TRUE)
print("Upload complete")